# BER curves

Reads the records under `results/ber` and plots them. Points resting on fewer than 30 error events are dropped; a zero-error point is drawn as a bound.

In [ ]:
import json, pathlib, re
import numpy as np
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().parent
BER = ROOT / "results" / "ber"
FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8, "legend.fontsize": 6.5, "xtick.labelsize": 7,
    "ytick.labelsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.3,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": ":",
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "pdf.fonttype": 42,
})

FLOOR = 30   # minimum error events for a point to be drawn

def curve(scheme):
    """Eb/N0, BER and the 95% interval for one scheme, ordered and filtered."""
    pts = []
    for f in (BER / scheme).glob("snr_*dB.json"):
        r = json.loads(f.read_text())
        if r["ers_cnt"] >= FLOOR or r["is_upper_bound"]:
            pts.append((r["eb_no_db"], r["ber"], r["ber_lo95"], r["ber_ub95"]))
    pts.sort()
    return [np.array(c) for c in zip(*pts)] if pts else None

In [ ]:
MARKER = {3: "o", 4: "s", 5: "^", 6: "D"}
RAMP = {"balanced": plt.get_cmap("Blues"), "unbalanced": plt.get_cmap("Oranges")}
SHADE = {3: 0.95, 4: 0.78, 5: 0.62, 6: 0.46}

fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.6), sharey=True)
for ax, fam in zip(axes, ("balanced", "unbalanced")):
    for L0 in (3, 4, 5, 6):
        c = curve(f"nsm_L{L0}_{fam}_conv_K3_7iters")
        if c is None:
            continue
        eb, ber, lo, hi = c
        ax.errorbar(eb, ber, yerr=[ber - lo, hi - ber], color=RAMP[fam](SHADE[L0]),
                    marker=MARKER[L0], markersize=3.5, elinewidth=0.7,
                    capsize=1.5, label=f"$L_0={L0}$")
    ax.set(yscale="log", xlabel="$E_b/N_0$ (dB)", title=fam,
           xlim=(0, 6), ylim=(1e-6, 0.5))
    ax.legend(loc="lower left")

axes[0].set_ylabel("BER")
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(FIGURES / f"ber_curves.{ext}")

for fam in ("balanced", "unbalanced"):
    for L0 in (3, 4, 5, 6):
        c = curve(f"nsm_L{L0}_{fam}_conv_K3_7iters")
        if c is not None:
            print(f"{fam:11s} L0={L0}  {len(c[0])} points  lowest BER {c[1].min():.2e}")